# Batch TRF Pipeline: Gammatone-8

Use this notebook to inspect and optionally launch the TRF-Tools batch job. By default it does not run the batch job.

In [3]:
from pathlib import Path
import subprocess
import sys

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_trf_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_trf_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_trf_experiment import PARAMETERS, alice
from jobs import JOBS

MODEL = 'gammatone-8'
RUN_BATCH = False  # Change to True only when you are ready to estimate TRFs for all subjects.

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'RUN_BATCH: {RUN_BATCH}')

INFO    :  *** AliceComprehensionTRF initialized with root /Users/yanyuwoo/Data/bids on 2026-07-08 17:06:11 ***
INFO    :  Using eelbrain 0.42.0a4, mne 1.11.0.
Pipeline directory: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline
RUN_BATCH: False


## Inspect Batch Definition

This confirms what the batch job will run before launching any computation.

In [4]:
subjects = alice.get_field_values('subject')
print(f'Subjects: {len(subjects)}')
print(subjects[:5], '...', subjects[-5:])
print('\nModel:', MODEL)
print('\nParameters:')
for key, value in PARAMETERS.items():
    print(f'  {key}: {value}')
print(f'\nNumber of JOBS entries: {len(JOBS)}')
JOBS

Subjects: 49
['01', '02', '03', '04', '05'] ... ['45', '46', '47', '48', '49']

Model: gammatone-8

Parameters:
  raw: 0.5-20
  samplingrate: 50
  data: eeg
  tstart: -0.1
  tstop: 1.0
  filter_x: continuous
  error: l1
  basis: 0.05
  partitions: -5
  selective_stopping: 1

Number of JOBS entries: 1


[<TRFsJob: gammatone-8, raw='0.5-20', samplingrate=50, data=TestDims('eeg'), tstart=-0.1, tstop=1.0, filter_x='continuous', error='l1', basis=0.05, partitions=-5, selective_stopping=1>]

## Preview Cache Targets

TRF-Tools writes results to its pipeline cache. This cell shows expected cache paths for a few subjects without fitting the model.

In [5]:
for subject in subjects[:5]:
    # make=True is needed for first-time model-name registration; path_only=True prevents fitting.
    path = alice.load_trf(MODEL, subject=subject, make=True, path_only=True, **PARAMETERS)
    print(subject, path)

01 /Users/yanyuwoo/Data/bids/derivatives/eelbrain/cache/trf/01/0.5-20/sub-01_eeg nobl -100-1000 50Hz gammatone-8 boosting h50 l1 con5ptns filtx=continuous ss1 cv.pickle
02 /Users/yanyuwoo/Data/bids/derivatives/eelbrain/cache/trf/02/0.5-20/sub-02_eeg nobl -100-1000 50Hz gammatone-8 boosting h50 l1 con5ptns filtx=continuous ss1 cv.pickle
03 /Users/yanyuwoo/Data/bids/derivatives/eelbrain/cache/trf/03/0.5-20/sub-03_eeg nobl -100-1000 50Hz gammatone-8 boosting h50 l1 con5ptns filtx=continuous ss1 cv.pickle
04 /Users/yanyuwoo/Data/bids/derivatives/eelbrain/cache/trf/04/0.5-20/sub-04_eeg nobl -100-1000 50Hz gammatone-8 boosting h50 l1 con5ptns filtx=continuous ss1 cv.pickle
05 /Users/yanyuwoo/Data/bids/derivatives/eelbrain/cache/trf/05/0.5-20/sub-05_eeg nobl -100-1000 50Hz gammatone-8 boosting h50 l1 con5ptns filtx=continuous ss1 cv.pickle


## Launch Batch Job

Run this only after the single-subject notebook succeeds. It calls the TRF-Tools job maker from the pipeline directory.

In [6]:
if RUN_BATCH:
    cmd = ['trf-tools-make-jobs', 'jobs.py']
    print('Running:', ' '.join(cmd))
    completed = subprocess.run(cmd, cwd=PIPELINE_DIR, check=False, text=True, capture_output=True)
    print('Return code:', completed.returncode)
    print('\nSTDOUT\n', completed.stdout)
    print('\nSTDERR\n', completed.stderr)
else:
    print('RUN_BATCH is False; batch job was not launched.')

RUN_BATCH is False; batch job was not launched.
